# Governed Banking Lakehouse — Medallion Architecture on Databricks

End-to-end data engineering pipeline over synthetic Canadian retail-banking data.
Raw CSVs land in a Unity Catalog Volume and pass through **Bronze → Silver → Gold**,
finishing as a governed star schema with **SCD Type 2** history, point-in-time correct
fact joins, an automated data-quality gate, streaming ingestion via Auto Loader, and
Change Data Feed for incremental downstream processing.

**Stack** — Databricks (serverless) · PySpark · Delta Lake · Unity Catalog · Spark SQL · Auto Loader

**Scale** — 2,000,000 transactions · 7,000 accounts · 5,000+ customers · 3 incremental batches

---

### Business context

| Consumer | Requirement | Design consequence |
|---|---|---|
| Risk | Customer attributes *as they were* at transaction time | SCD Type 2 on `dim_customer` |
| Marketing | One accurate row per customer (Customer 360) | Deduplication + survivorship rules |
| Finance | Monthly spend by region | Point-in-time join on the fact table |

Overwriting a customer's city nightly would silently corrupt history: a January transaction
would be attributed to the city they moved to in March. That requirement drove the entire
Gold-layer design, and section 11 quantifies what it is worth.

## 1. Setup — catalog, schema and landing volume

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS banking")
spark.sql("CREATE SCHEMA IF NOT EXISTS banking.raw")
spark.sql("CREATE VOLUME IF NOT EXISTS banking.raw.landing")
print("ok")

ok


### 1.1 Generate raw source data

Synthetic CSVs written to a Unity Catalog Volume with data-quality problems introduced
deliberately: duplicate customer IDs, empty vs null emails, inconsistent city casing, two
date formats, orphan account references, and negative/zero amounts.

Three batches simulate daily CRM deltas. Batches 2 and 3 contain customers who changed
city — the input SCD Type 2 exists to capture.

In [0]:
import csv, os, random
from datetime import datetime, timedelta

RAW = "/Volumes/banking/raw/landing"
random.seed(42)

N_CUSTOMERS = 5000
N_ACCOUNTS  = 7000
N_TXNS      = 2000000

CITIES = ["Ottawa", "Toronto", "Montreal", "Calgary", "Vancouver", "Halifax"]
PROVINCES = {"Ottawa":"ON","Toronto":"ON","Montreal":"QC",
             "Calgary":"AB","Vancouver":"BC","Halifax":"NS"}
ACCOUNT_TYPES = ["CHEQUING","SAVINGS","CREDIT","MORTGAGE"]
TXN_TYPES = ["PURCHASE","WITHDRAWAL","DEPOSIT","TRANSFER","PAYMENT"]
MERCHANTS = ["Amazon","Tim Hortons","Loblaws","Shell","Costco",
             "Canadian Tire","Uber","Netflix","Sobeys","Home Depot"]
FIRST = ["Jonathan","Jodi","James","Alexis","Brittany","Katelyn","Kimberly",
         "Monica","Daniel","Priya","Ahmed","Wei","Sofia","Liam","Noah","Emma"]
LAST  = ["Cox","Miller","Woods","Smith","Osborne","James","Ortiz","Sanchez",
         "Tremblay","Patel","Khan","Chen","Nguyen","Brown","Wilson","Roy"]

BASE = datetime(2024, 1, 1)

def dirty_city(city):
    r = random.random()
    if r < 0.15: return city.upper()
    if r < 0.30: return city.lower()
    if r < 0.40: return f"  {city} "
    return city

def dirty_date(d):
    return d.strftime("%d/%m/%Y") if random.random() < 0.15 else d.strftime("%Y-%m-%d")

def make_customer(cid, updated_at, city):
    first, last = random.choice(FIRST), random.choice(LAST)
    email = f"{first.lower()}.{last.lower()}@example.com"
    r = random.random()
    if r < 0.08: email = ""
    elif r < 0.14: email = None
    phone = f"{random.randint(200,999)}-{random.randint(200,999)}-{random.randint(1000,9999)}"
    if random.random() < 0.10: phone = None
    return {"customer_id": cid, "first_name": first, "last_name": last,
            "email": email, "phone": phone, "city": dirty_city(city),
            "province": PROVINCES[city],
            "updated_at": updated_at.strftime("%Y-%m-%d %H:%M:%S")}

def write_csv(path, rows, fields):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if v is None else v) for k, v in r.items()})

CUST_FIELDS = ["customer_id","first_name","last_name","email","phone","city","province","updated_at"]
ACCT_FIELDS = ["account_id","customer_id","account_type","status","opened_date","branch_id","updated_at"]
TXN_FIELDS  = ["txn_id","account_id","txn_timestamp","amount","txn_type","merchant","currency"]

# branches
branches, BRANCH_IDS = [], []
for i, city in enumerate(CITIES + CITIES, start=1):
    bid = f"BR{i:03d}"
    suffix = "Main" if i <= 6 else "West"
    branches.append({"branch_id":bid, "branch_name":f"{city} {suffix}",
                     "city":dirty_city(city), "province":PROVINCES[city]})
    BRANCH_IDS.append(bid)
write_csv(f"{RAW}/batch_1/branches.csv", branches, ["branch_id","branch_name","city","province"])

# customers batch 1
customer_city, customers_b1 = {}, []
for i in range(1, N_CUSTOMERS+1):
    cid = f"C{i:06d}"
    city = random.choice(CITIES)
    customer_city[cid] = city
    upd = BASE + timedelta(days=random.randint(0,30), hours=random.randint(0,23))
    customers_b1.append(make_customer(cid, upd, city))

for d in random.sample(customers_b1, int(N_CUSTOMERS*0.05)):
    later = datetime.strptime(d["updated_at"], "%Y-%m-%d %H:%M:%S") + timedelta(hours=random.randint(1,8))
    customers_b1.append(make_customer(d["customer_id"], later, customer_city[d["customer_id"]]))

random.shuffle(customers_b1)
write_csv(f"{RAW}/batch_1/customers.csv", customers_b1, CUST_FIELDS)

# accounts
accounts, account_ids = [], []
for i in range(1, N_ACCOUNTS+1):
    aid = f"A{i:07d}"
    accounts.append({"account_id":aid, "customer_id":f"C{random.randint(1,N_CUSTOMERS):06d}",
        "account_type":random.choice(ACCOUNT_TYPES),
        "status":random.choices(["ACTIVE","CLOSED","DORMANT"], weights=[80,10,10])[0],
        "opened_date":dirty_date(BASE - timedelta(days=random.randint(30,2000))),
        "branch_id":random.choice(BRANCH_IDS),
        "updated_at":(BASE + timedelta(days=random.randint(0,30))).strftime("%Y-%m-%d %H:%M:%S")})
    account_ids.append(aid)
write_csv(f"{RAW}/batch_1/accounts.csv", accounts, ACCT_FIELDS)

# transactions
txns = []
for i in range(1, N_TXNS+1):
    aid = f"A9{random.randint(100000,999999)}" if random.random() < 0.005 else random.choice(account_ids)
    ts = BASE + timedelta(days=random.randint(0,89), hours=random.randint(0,23), minutes=random.randint(0,59))
    amount = round(random.lognormvariate(3.2, 1.1), 2)
    r = random.random()
    if r < 0.003: amount = -amount
    elif r < 0.005: amount = 0.0
    txns.append({"txn_id":f"T{i:09d}", "account_id":aid,
        "txn_timestamp":ts.strftime("%Y-%m-%d %H:%M:%S"), "amount":amount,
        "txn_type":random.choice(TXN_TYPES),
        "merchant":random.choice(MERCHANTS) if random.random() > 0.05 else "",
        "currency":"CAD"})
write_csv(f"{RAW}/batch_1/transactions.csv", txns, TXN_FIELDS)

# batch 2 and 3
def make_batch(day, n_movers, new_start, n_new):
    out, when = [], BASE + timedelta(days=day)
    for cid in random.sample(list(customer_city.keys()), n_movers):
        new_city = random.choice([c for c in CITIES if c != customer_city[cid]])
        customer_city[cid] = new_city
        out.append(make_customer(cid, when + timedelta(hours=random.randint(0,23)), new_city))
    for i in range(new_start, new_start + n_new):
        cid = f"C{i:06d}"
        city = random.choice(CITIES)
        customer_city[cid] = city
        out.append(make_customer(cid, when + timedelta(hours=random.randint(0,23)), city))
    random.shuffle(out)
    return out

write_csv(f"{RAW}/batch_2/customers.csv", make_batch(45, 500, N_CUSTOMERS+1, 200), CUST_FIELDS)
write_csv(f"{RAW}/batch_3/customers.csv", make_batch(75, 300, N_CUSTOMERS+201, 150), CUST_FIELDS)

print("generated")

generated


In [0]:
dbutils.fs.ls("/Volumes/banking/raw/landing/batch_1")

[FileInfo(path='dbfs:/Volumes/banking/raw/landing/batch_1/accounts.csv', name='accounts.csv', size=492649, modificationTime=1790029546000),
 FileInfo(path='dbfs:/Volumes/banking/raw/landing/batch_1/branches.csv', name='branches.csv', size=417, modificationTime=1790029546000),
 FileInfo(path='dbfs:/Volumes/banking/raw/landing/batch_1/customers.csv', name='customers.csv', size=455696, modificationTime=1790029546000),
 FileInfo(path='dbfs:/Volumes/banking/raw/landing/batch_1/transactions.csv', name='transactions.csv', size=135830695, modificationTime=1790029565000)]

## 2. Data profiling — finding the problems

Nothing is transformed before the source is understood. Each finding below drove a specific
design decision later in the pipeline.

In [0]:
BASE = "/Volumes/banking/raw/landing"

raw_cust   = spark.read.csv(f"{BASE}/batch_1/customers.csv", header=True)
raw_acct   = spark.read.csv(f"{BASE}/batch_1/accounts.csv", header=True)
raw_txn    = spark.read.csv(f"{BASE}/batch_1/transactions.csv", header=True)
raw_branch = spark.read.csv(f"{BASE}/batch_1/branches.csv", header=True)

for name, df in [("customers", raw_cust), ("accounts", raw_acct),
                 ("transactions", raw_txn), ("branches", raw_branch)]:
    print(f"\n===== {name} — {df.count():,} rows =====")
    print(df.columns)
    df.show(3, truncate=False)


===== customers — 5,250 rows =====
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'province', 'updated_at']
+-----------+----------+---------+-----------------------------+------------+-------+--------+-------------------+
|customer_id|first_name|last_name|email                        |phone       |city   |province|updated_at         |
+-----------+----------+---------+-----------------------------+------------+-------+--------+-------------------+
|C001344    |Katelyn   |James    |NULL                         |706-946-2957|HALIFAX|NS      |2024-01-07 07:00:00|
|C004897    |Jonathan  |Nguyen   |jonathan.nguyen@example.com  |223-481-3303|Toronto|ON      |2024-01-19 04:00:00|
|C003786    |Kimberly  |Tremblay |kimberly.tremblay@example.com|NULL        |ottawa |ON      |2024-01-12 18:00:00|
+-----------+----------+---------+-----------------------------+------------+-------+--------+-------------------+
only showing top 3 rows

===== accounts — 7,000 rows =====
['acc

### 2.1 Is the primary key actually unique?

In [0]:
raw_cust = spark.read.csv(
    "/Volumes/banking/raw/landing/batch_1/customers.csv",
    header=True
)

raw_cust.printSchema()
raw_cust.show(10, truncate=False)
print(raw_cust.count())

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- province: string (nullable = true)
 |-- updated_at: string (nullable = true)

+-----------+----------+---------+-----------------------------+------------+--------+--------+-------------------+
|customer_id|first_name|last_name|email                        |phone       |city    |province|updated_at         |
+-----------+----------+---------+-----------------------------+------------+--------+--------+-------------------+
|C001344    |Katelyn   |James    |NULL                         |706-946-2957|HALIFAX |NS      |2024-01-07 07:00:00|
|C004897    |Jonathan  |Nguyen   |jonathan.nguyen@example.com  |223-481-3303|Toronto |ON      |2024-01-19 04:00:00|
|C003786    |Kimberly  |Tremblay |kimberly.tremblay@example.com|NULL        |ottawa  |ON      |2

In [0]:
from pyspark.sql import functions as F

print("total rows:      ", raw_cust.count())
print("distinct ids:    ", raw_cust.select("customer_id").distinct().count())

raw_cust.groupBy("customer_id").count().filter(F.col("count") > 1).show(5)

total rows:       5250
distinct ids:     5000
+-----------+-----+
|customer_id|count|
+-----------+-----+
|    C000121|    2|
|    C002910|    2|
|    C001686|    2|
|    C001438|    2|
|    C004402|    2|
+-----------+-----+
only showing top 5 rows


**Finding:** 5,250 rows for 5,000 distinct customer IDs — 250 duplicates. Inspecting one reveals this is *not* a simple duplicate.

In [0]:
raw_cust.filter(F.col("customer_id") == "C000121").show(truncate=False)

+-----------+----------+---------+--------------------------+------------+---------+--------+-------------------+
|customer_id|first_name|last_name|email                     |phone       |city     |province|updated_at         |
+-----------+----------+---------+--------------------------+------------+---------+--------+-------------------+
|C000121    |Emma      |Khan     |NULL                      |720-803-8935|vancouver|BC      |2024-01-12 08:00:00|
|C000121    |Daniel    |Sanchez  |daniel.sanchez@example.com|774-985-1430|Vancouver|BC      |2024-01-12 03:00:00|
+-----------+----------+---------+--------------------------+------------+---------+--------+-------------------+



**Finding:** the two rows carry entirely different names. This is an **ID collision** — two
people sharing an identifier — not one customer updated twice.

That distinction matters. Survivorship rules (borrowing a missing email from the older row)
are only valid *within a confirmed entity*. Merging fields across two different people
fabricates a customer who does not exist, which is worse than leaving a NULL.

### 2.2 Referential integrity — do all transactions belong to a real account?

In [0]:
raw_txn = spark.read.csv("/Volumes/banking/raw/landing/batch_1/transactions.csv", header=True)
raw_acct = spark.read.csv("/Volumes/banking/raw/landing/batch_1/accounts.csv", header=True)

raw_txn.printSchema()
raw_acct.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_timestamp: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- currency: string (nullable = true)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- opened_date: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- updated_at: string (nullable = true)



In [0]:
orphans = raw_txn.join(raw_acct, "account_id", "left_anti")
print("orphan transactions:", orphans.count())

orphan transactions: 10035


**Finding:** 10,035 orphan transactions (~0.5%).

**Decision: flag, do not drop.** Accounts and transactions arrive from different source
systems on different schedules, so an orphan is usually a *late-arriving dimension* rather
than corrupt data. Dropping the row loses it permanently even after the account appears.
Instead the row is retained with `is_orphan = true` and the orphan *rate* is monitored — a
jump from 0.5% to 20% means the accounts feed failed, not that customers changed behaviour.

## 3. Bronze — raw landing

Bronze stores source data **exactly as it arrived**: no deduplication, no cleaning, no type
casting. Only three audit columns are added.

Keeping the raw form enables replay (fix Silver logic and reprocess without asking the source
to resend) and audit (trace any value back to the file and batch that produced it).

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS banking.bronze")
print("ok")

ok


In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/banking/raw/landing"

raw_cust   = spark.read.csv(f"{BASE}/batch_1/customers.csv", header=True)
raw_acct   = spark.read.csv(f"{BASE}/batch_1/accounts.csv", header=True)
raw_txn    = spark.read.csv(f"{BASE}/batch_1/transactions.csv", header=True)
raw_branch = spark.read.csv(f"{BASE}/batch_1/branches.csv", header=True)

print(raw_cust.count())

5250


In [0]:
from pyspark.sql import functions as F

bronze_cust = (
    raw_cust
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("customers.csv"))
    .withColumn("batch_id", F.lit("batch_1"))
)

bronze_cust.write.mode("overwrite").saveAsTable("banking.bronze.customers")
print("written")

written


In [0]:
def to_bronze(df, filename, table_name, batch="batch_1"):
    (df
     .withColumn("ingestion_timestamp", F.current_timestamp())
     .withColumn("source_file", F.lit(filename))
     .withColumn("batch_id", F.lit(batch))
     .write.mode("overwrite").saveAsTable(f"banking.bronze.{table_name}"))
    print(f"{table_name} written")

to_bronze(raw_acct,   "accounts.csv",     "accounts")
to_bronze(raw_txn,    "transactions.csv", "transactions")
to_bronze(raw_branch, "branches.csv",     "branches")

accounts written
transactions written
branches written


In [0]:
spark.sql("SHOW TABLES IN banking.bronze").show()

spark.sql("SELECT * FROM banking.bronze.customers LIMIT 5").show(truncate=False)

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
|  bronze|    accounts|      false|
|  bronze|    branches|      false|
|  bronze|   customers|      false|
|  bronze|transactions|      false|
|        |      staged|       true|
+--------+------------+-----------+

+-----------+----------+---------+-----------------------------+------------+--------+--------+-------------------+--------------------------+-------------+--------+
|customer_id|first_name|last_name|email                        |phone       |city    |province|updated_at         |ingestion_timestamp       |source_file  |batch_id|
+-----------+----------+---------+-----------------------------+------------+--------+--------+-------------------+--------------------------+-------------+--------+
|C001344    |Katelyn   |James    |NULL                         |706-946-2957|HALIFAX |NS      |2024-01-07 07:00:00|2026-09-21 22:26:46.459373|customers.csv|batch_1 |
|C004897   

### 3.1 Delta transaction log

Delta is Parquet plus a transaction log. The log gives atomic writes (readers never see a
half-written table), time travel to earlier versions, and schema enforcement.

In [0]:
spark.sql("DESCRIBE HISTORY banking.bronze.customers").show(truncate=False)

+-------+-------------------+--------------+-------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                 |operation                        |operationParameters                                                 

## 4. Silver — cleaned and conformed

Silver enforces *one row per real entity*, correct types, and standardised values. Problems
are **flagged, not deleted** — Silver makes issues visible; the quality gate and downstream
consumers decide what to exclude.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS banking.silver")
print("ok")

ok


### 4.1 Customers — deduplication

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_cust = spark.table("banking.bronze.customers")

w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

deduped = (
    bronze_cust
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("before:", bronze_cust.count())
print("after: ", deduped.count())

before: 5250
after:  5000


In [0]:
deduped.filter(F.col("customer_id") == "C000121").select(
    "customer_id", "first_name", "email", "phone", "updated_at"
).show(truncate=False)

+-----------+----------+-----+------------+-------------------+
|customer_id|first_name|email|phone       |updated_at         |
+-----------+----------+-----+------------+-------------------+
|C000121    |Emma      |NULL |720-803-8935|2024-01-12 08:00:00|
+-----------+----------+-----+------------+-------------------+



**Finding:** the surviving row for C000121 has a NULL email while the discarded row had a
valid one — the case that normally calls for field-level survivorship. Before applying it,
confirm the rows describe the same person.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_cust = spark.table("banking.bronze.customers")

w_full = Window.partitionBy("customer_id").rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing
)

with_names = bronze_cust.withColumn(
    "name_count",
    F.size(F.collect_set(F.concat_ws("|", "first_name", "last_name")).over(w_full))
)

conflicts = with_names.filter(F.col("name_count") > 1).select("customer_id").distinct()

print("ids with conflicting names:", conflicts.count())

ids with conflicting names: 250


**Finding:** all 250 duplicated IDs have conflicting names. Every duplicate is an ID
collision, so **no survivorship is applied**.

**Resolution:**
- keep the most recent row per ID in `silver.customers`, marked `is_id_conflict = true`
- preserve the losing row in `silver.customer_conflicts` with a reason, rather than deleting it
- row counts must reconcile: 4,750 clean + 250 kept + 250 quarantined = 5,250 ✓

In production this is escalated to the CRM owner — 5% of the customer base sharing
identifiers is a source-system defect, not something to silently patch downstream.

In [0]:
w_order = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

ranked = (
    with_names
    .withColumn("rn", F.row_number().over(w_order))
    .withColumn("is_id_conflict", F.col("name_count") > 1)
)

ranked.groupBy("rn", "is_id_conflict").count().orderBy("rn").show()

+---+--------------+-----+
| rn|is_id_conflict|count|
+---+--------------+-----+
|  1|         false| 4750|
|  1|          true|  250|
|  2|          true|  250|
+---+--------------+-----+



In [0]:
silver_cust_stage = ranked.filter(F.col("rn") == 1).drop("rn", "name_count")

conflicts_out = (
    ranked
    .filter(F.col("rn") > 1)
    .drop("rn", "name_count")
    .withColumn("conflict_reason", F.lit("lost id collision to newer record"))
)

print("silver:   ", silver_cust_stage.count())
print("conflicts:", conflicts_out.count())

silver:    5000
conflicts: 250


### 4.2 Customers — standardisation and typing

In [0]:
silver_cust_stage = (
    silver_cust_stage
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name",  F.initcap(F.trim(F.col("last_name"))))
    .withColumn("city",       F.upper(F.trim(F.col("city"))))
    .withColumn("email",      F.lower(F.trim(F.col("email"))))
    .withColumn("email",      F.when(F.col("email") == "", None).otherwise(F.col("email")))
    .withColumn("updated_at", F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"))
)

silver_cust_stage.groupBy("city").count().orderBy("city").show()

+---------+-----+
|     city|count|
+---------+-----+
|  CALGARY|  838|
|  HALIFAX|  837|
| MONTREAL|  849|
|   OTTAWA|  810|
|  TORONTO|  852|
|VANCOUVER|  814|
+---------+-----+



In [0]:
silver_cust_stage.select(
    F.count("*").alias("total"),
    F.count("email").alias("has_email"),
    F.sum(F.when(F.col("email").isNull(), 1).otherwise(0)).alias("null_email")
).show()

+-----+---------+----------+
|total|has_email|null_email|
+-----+---------+----------+
| 5000|     4333|       667|
+-----+---------+----------+



**Finding:** 667 of 5,000 customers (13.3%) have no email address.

Empty strings are converted to true NULLs rather than left as `''` or replaced with
`'unknown'` — an invented value looks like data and defeats null checks. The gap is reported
as a metric for Marketing to act on.

In [0]:
silver_cust_stage.write.mode("overwrite").saveAsTable("banking.silver.customers")
conflicts_out.write.mode("overwrite").saveAsTable("banking.silver.customer_conflicts")

print("customers:", spark.table("banking.silver.customers").count())
print("conflicts:", spark.table("banking.silver.customer_conflicts").count())

customers: 5000
conflicts: 250


### 4.3 Accounts — multiple date formats

`opened_date` arrives in two formats. Databricks runs in ANSI mode, so a strict parse fails
the whole job rather than silently returning nulls.

In [0]:
bronze_acct.select(
    "opened_date",
    F.to_date("opened_date", "yyyy-MM-dd").alias("parsed")
).show(10)

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6900438107506598>, line 4
      1 bronze_acct.select(
      2     "opened_date",
      3     F.to_date("opened_date", "yyyy-MM-dd").alias("parsed")
----> 4 ).show(10)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1200, in DataFrame.show(self, n, truncate, vertical)
   1199 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1200     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:911, in DataFrame._show_string(self, n, truncate, vertical)
    894     except ValueError:
    895         raise PySparkTypeError(
    896             errorClass="NOT_BOOL",
    897             messageParameters={
   (...)
    900             },
    901         )
    903 tabl

The failure above is the desired behaviour — a silent null would have lost ~15% of dates
unnoticed. The fix is to try each known format and take the first that succeeds.

In [0]:
bronze_acct.select(
    "opened_date",
    F.coalesce(
        F.try_to_date("opened_date", "yyyy-MM-dd"),
        F.try_to_date("opened_date", "dd/MM/yyyy")
    ).alias("parsed")
).show(10)

+-----------+----------+
|opened_date|    parsed|
+-----------+----------+
| 2023-09-25|2023-09-25|
| 2022-07-15|2022-07-15|
| 27/10/2021|2021-10-27|
| 2020-04-06|2020-04-06|
| 2019-11-25|2019-11-25|
| 2021-07-09|2021-07-09|
| 2018-08-28|2018-08-28|
| 2023-03-22|2023-03-22|
| 2022-07-08|2022-07-08|
| 2021-01-04|2021-01-04|
+-----------+----------+
only showing top 10 rows


### 4.4 Proving the format list is complete

Assuming two formats is not the same as knowing. Any value matching neither parses to NULL,
so counting *text present but parse failed* proves coverage.

In [0]:
parsed = bronze_acct.withColumn(
    "parsed",
    F.coalesce(
        F.try_to_date("opened_date", "yyyy-MM-dd"),
        F.try_to_date("opened_date", "dd/MM/yyyy")
    )
)

failed = parsed.filter(F.col("parsed").isNull() & F.col("opened_date").isNotNull())

print("could not parse:", failed.count())
failed.select("opened_date").distinct().show()

could not parse: 0
+-----------+
|opened_date|
+-----------+
+-----------+



In [0]:
check = bronze_acct.withColumn(
    "ts", F.try_to_timestamp("updated_at", F.lit("yyyy-MM-dd HH:mm:ss"))
)

bad = check.filter(F.col("ts").isNull() & F.col("updated_at").isNotNull())
print("could not parse updated_at:", bad.count())

**Result: 0 unparseable values** — coverage proven, not assumed.

*Caveat:* `09/10/2018` is ambiguous between `dd/MM` and `MM/dd` and parses silently either
way. Format direction must be confirmed with the source system owner; no amount of profiling
can resolve it.

In [0]:
silver_acct = (
    bronze_acct
    .withColumn("opened_date", F.coalesce(
        F.try_to_date("opened_date", "yyyy-MM-dd"),
        F.try_to_date("opened_date", "dd/MM/yyyy")))
    .withColumn("account_type", F.upper(F.trim(F.col("account_type"))))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .withColumn("updated_at", F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"))
)

silver_acct.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- opened_date: date (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [0]:
silver_acct.write.mode("overwrite").saveAsTable("banking.silver.accounts")
print("accounts:", spark.table("banking.silver.accounts").count())

accounts: 7000


### 4.5 Transactions — typing and flagging

2M rows. Casts are validated before being applied, then two flags are added: `is_orphan`
(no matching account) and `is_invalid_amount` (zero or negative).

In [0]:
bronze_txn = spark.table("banking.bronze.transactions")

typed = (
    bronze_txn
    .withColumn("amount_num", F.col("amount").try_cast("double"))
    .withColumn("txn_ts", F.try_to_timestamp("txn_timestamp", F.lit("yyyy-MM-dd HH:mm:ss")))
)

print("bad amount:   ", typed.filter(F.col("amount_num").isNull() & F.col("amount").isNotNull()).count())
print("bad timestamp:", typed.filter(F.col("txn_ts").isNull() & F.col("txn_timestamp").isNotNull()).count())

bad amount:    0
bad timestamp: 0


In [0]:
from pyspark.sql import functions as F

bronze_txn  = spark.table("banking.bronze.transactions")
silver_acct = spark.table("banking.silver.accounts")

account_list = silver_acct.select("account_id").withColumn("acct_exists", F.lit(True))

silver_txn = (
    bronze_txn
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("txn_timestamp", F.to_timestamp("txn_timestamp", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("txn_date", F.to_date("txn_timestamp"))
    .withColumn("merchant", F.when(F.trim(F.col("merchant")) == "", None).otherwise(F.col("merchant")))
    .withColumn("is_invalid_amount", F.col("amount") <= 0)
    .join(account_list, "account_id", "left")
    .withColumn("is_orphan", F.col("acct_exists").isNull())
    .drop("acct_exists")
)

silver_txn.groupBy("is_orphan", "is_invalid_amount").count().show()

+---------+-----------------+-------+
|is_orphan|is_invalid_amount|  count|
+---------+-----------------+-------+
|     true|            false|   9984|
|    false|            false|1980047|
|    false|             true|   9918|
|     true|             true|     51|
+---------+-----------------+-------+



**Findings:** 10,035 orphans, 9,969 invalid amounts, and **51 rows that are both** — the
most likely candidates for genuine corruption rather than timing lag.

Written partitioned by `txn_date`. At 2M rows the partitions are small; production guidance
is to partition only above ~1TB and use liquid clustering below that. Included here to
demonstrate partition pruning.

In [0]:
silver_txn.write.mode("overwrite").partitionBy("txn_date").saveAsTable("banking.silver.transactions")

print("transactions:", spark.table("banking.silver.transactions").count())

transactions: 2000000


### 4.6 Branches

In [0]:
from pyspark.sql import functions as F

bronze_br = spark.table("banking.bronze.branches")

silver_br = (
    bronze_br
    .withColumn("city", F.upper(F.trim(F.col("city"))))
    .withColumn("branch_name", F.trim(F.col("branch_name")))
)

silver_br.write.mode("overwrite").saveAsTable("banking.silver.branches")
spark.table("banking.silver.branches").show()

+---------+--------------+---------+--------+--------------------+------------+--------+
|branch_id|   branch_name|     city|province| ingestion_timestamp| source_file|batch_id|
+---------+--------------+---------+--------+--------------------+------------+--------+
|    BR001|   Ottawa Main|   OTTAWA|      ON|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR002|  Toronto Main|  TORONTO|      ON|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR003| Montreal Main| MONTREAL|      QC|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR004|  Calgary Main|  CALGARY|      AB|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR005|Vancouver Main|VANCOUVER|      BC|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR006|  Halifax Main|  HALIFAX|      NS|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR007|   Ottawa West|   OTTAWA|      ON|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR008|  Toronto West|  TORONTO|      ON|2026-09-21 22:26:...|branches.csv| batch_1|
|    BR009| Montreal 

## 5. Gold — dimensional model with SCD Type 2

`dim_customer` tracks history as additional rows rather than overwriting. Each version
carries `valid_from`, `valid_to`, `is_current`, and a surrogate key `customer_sk` — necessary
because `customer_id` stops being unique once history exists.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS banking.gold")
print("ok")

ok


In [0]:
spark.table("banking.silver.customers").printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- province: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- is_id_conflict: boolean (nullable = true)



### 5.1 Initial load

In [0]:
from pyspark.sql import functions as F

silver_cust = spark.table("banking.silver.customers")

dim_customer = (
    silver_cust
    .select(
        "customer_id", "first_name", "last_name", "email", "phone",
        "city", "province", "is_id_conflict",
        F.col("updated_at").alias("valid_from"),
        F.to_timestamp(F.lit("9999-12-31 00:00:00")).alias("valid_to"),
        F.lit(True).alias("is_current"),
    )
    .withColumn("customer_sk", F.xxhash64("customer_id", "valid_from"))
)

dim_customer.write.mode("overwrite").saveAsTable("banking.gold.dim_customer")

spark.table("banking.gold.dim_customer").filter(F.col("customer_id") == "C002632").show(truncate=False)

+-----------+----------+---------+--------------------------+------------+-------+--------+--------------+-------------------+-------------------+----------+-------------------+
|customer_id|first_name|last_name|email                     |phone       |city   |province|is_id_conflict|valid_from         |valid_to           |is_current|customer_sk        |
+-----------+----------+---------+--------------------------+------------+-------+--------+--------------+-------------------+-------------------+----------+-------------------+
|C002632    |Jonathan  |Ortiz    |jonathan.ortiz@example.com|606-287-7474|CALGARY|AB      |false         |2024-01-17 22:00:00|9999-12-31 00:00:00|true      |2088472562978550675|
+-----------+----------+---------+--------------------------+------------+-------+--------+--------------+-------------------+-------------------+----------+-------------------+



### 5.2 Batch 2 — classify each incoming row

Incoming rows are cleaned with **identical rules** to batch 1. Any difference in cleaning
would make standardisation look like a real change and corrupt the history.

In [0]:
from pyspark.sql import functions as F

b2 = (
    spark.read.csv("/Volumes/banking/raw/landing/batch_2/customers.csv", header=True)
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name",  F.initcap(F.trim(F.col("last_name"))))
    .withColumn("city",       F.upper(F.trim(F.col("city"))))
    .withColumn("email",      F.lower(F.trim(F.col("email"))))
    .withColumn("email",      F.when(F.col("email") == "", None).otherwise(F.col("email")))
    .withColumn("updated_at", F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"))
)

print("batch 2 rows:", b2.count())
print("distinct ids:", b2.select("customer_id").distinct().count())

batch 2 rows: 700
distinct ids: 700


In [0]:
from pyspark.sql import functions as F

dim = spark.table("banking.gold.dim_customer").filter(F.col("is_current") == True)

compared = (
    b2.alias("new")
    .join(dim.alias("cur"), "customer_id", "left")
    .withColumn("change_type",
        F.when(F.col("cur.city").isNull(), "NEW")
         .when(F.col("new.city") != F.col("cur.city"), "CHANGED")
         .otherwise("UNCHANGED"))
)

compared.groupBy("change_type").count().show()

+-----------+-----+
|change_type|count|
+-----------+-----+
|    CHANGED|  500|
|        NEW|  200|
+-----------+-----+



**Result:** 500 CHANGED, 200 NEW.

`NEW` must be tested before `CHANGED`: comparing a value to NULL yields unknown, never true,
so a new customer would otherwise fall through to UNCHANGED.

### 5.3 The two-copy staging pattern

`MERGE` performs **one action per source row**, but SCD Type 2 requires **two** for a changed
customer: close the current row *and* insert a new version.

The solution is to stage each changed customer twice:

| copy | `merge_key` | matches? | effect |
|---|---|---|---|
| 1 | `customer_id` | yes | UPDATE — closes the current row |
| 2 | `NULL` | never | INSERT — writes the new version |

NULL never equals anything, guaranteeing copy 2 falls into `WHEN NOT MATCHED`.

In [0]:
from pyspark.sql import functions as F

new_rows = compared.select("new.*", "change_type")

to_close = (
    new_rows.filter(F.col("change_type") == "CHANGED")
    .withColumn("merge_key", F.col("customer_id"))
)

to_insert = (
    new_rows.filter(F.col("change_type").isin("CHANGED", "NEW"))
    .withColumn("merge_key", F.lit(None).cast("string"))
)

staged = to_close.unionByName(to_insert)

staged.groupBy("change_type", F.col("merge_key").isNull().alias("key_is_null")).count().show()

+-----------+-----------+-----+
|change_type|key_is_null|count|
+-----------+-----------+-----+
|    CHANGED|      false|  500|
|    CHANGED|       true|  500|
|        NEW|       true|  200|
+-----------+-----------+-----+



In [0]:
staged.createOrReplaceTempView("staged")

spark.sql("""
MERGE INTO banking.gold.dim_customer AS t
USING staged AS s
ON t.customer_id = s.merge_key AND t.is_current = true

WHEN MATCHED THEN UPDATE SET
    t.is_current = false,
    t.valid_to   = s.updated_at

WHEN NOT MATCHED THEN INSERT (
    customer_id, first_name, last_name, email, phone, city, province,
    is_id_conflict, valid_from, valid_to, is_current, customer_sk
) VALUES (
    s.customer_id, s.first_name, s.last_name, s.email, s.phone, s.city, s.province,
    false, s.updated_at, timestamp'9999-12-31 00:00:00', true,
    xxhash64(s.customer_id, s.updated_at)
)
""")

print("dim rows now:", spark.table("banking.gold.dim_customer").count())

dim rows now: 5700


### 5.4 Verifying the history chain

In [0]:
from pyspark.sql import functions as F

dim = spark.table("banking.gold.dim_customer")

mover = (dim.groupBy("customer_id").count()
            .filter(F.col("count") > 1)
            .first()["customer_id"])

print("checking:", mover)

(dim.filter(F.col("customer_id") == mover)
    .select("customer_id", "city", "valid_from", "valid_to", "is_current")
    .orderBy("valid_from")
    .show(truncate=False))

checking: C002523
+-----------+--------+-------------------+-------------------+----------+
|customer_id|city    |valid_from         |valid_to           |is_current|
+-----------+--------+-------------------+-------------------+----------+
|C002523    |OTTAWA  |2024-01-04 11:00:00|2024-02-15 23:00:00|false     |
|C002523    |MONTREAL|2024-02-15 23:00:00|9999-12-31 00:00:00|true      |
+-----------+--------+-------------------+-------------------+----------+



Each version's `valid_to` equals the next version's `valid_from` — no gap (a moment with no
known city) and no overlap (a moment with two). The range join uses
`>= valid_from AND < valid_to` so a transaction at the exact boundary matches one version
only.

In [0]:
from pyspark.sql import functions as F

dim = spark.table("banking.gold.dim_customer")

bad = (dim.filter(F.col("is_current"))
          .groupBy("customer_id").count()
          .filter(F.col("count") != 1))

print("customers with wrong current count:", bad.count())
print("total customers:", dim.select("customer_id").distinct().count())
print("current rows:   ", dim.filter(F.col("is_current")).count())

customers with wrong current count: 0
total customers: 5200
current rows:    5200


**Integrity check:** exactly one current row per customer. Zero would orphan the customer;
two would duplicate every downstream join.

## 6. Incremental loading

The batch-2 steps are wrapped in a reusable function so each subsequent delta is applied on
top of existing state — no full reload.

`localCheckpoint()` materialises the staged rows before the MERGE. Without it the staging
DataFrame is a lazy plan that recomputes against the very table the MERGE is modifying,
producing different results on re-evaluation.

In [0]:
from pyspark.sql import functions as F

def load_customer_batch(path):
    # 1. read and clean — same rules as batch 1
    new = (
        spark.read.csv(path, header=True)
        .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
        .withColumn("last_name",  F.initcap(F.trim(F.col("last_name"))))
        .withColumn("city",       F.upper(F.trim(F.col("city"))))
        .withColumn("email",      F.lower(F.trim(F.col("email"))))
        .withColumn("email",      F.when(F.col("email") == "", None).otherwise(F.col("email")))
        .withColumn("updated_at", F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"))
    )

    # 2. compare to current rows
    cur = spark.table("banking.gold.dim_customer").filter(F.col("is_current"))
    compared = (
        new.alias("new").join(cur.alias("cur"), "customer_id", "left")
        .withColumn("change_type",
            F.when(F.col("cur.city").isNull(), "NEW")
             .when(F.col("new.city") != F.col("cur.city"), "CHANGED")
             .otherwise("UNCHANGED"))
        .select("new.*", "change_type")
    )

    # 3. build the two copies
    to_close  = compared.filter(F.col("change_type") == "CHANGED").withColumn("merge_key", F.col("customer_id"))
    to_insert = compared.filter(F.col("change_type").isin("CHANGED", "NEW")).withColumn("merge_key", F.lit(None).cast("string"))
    staged = to_close.unionByName(to_insert)

    # 4. freeze it — so the MERGE can't change its own input
    staged = staged.localCheckpoint()
    staged.groupBy("change_type").count().show()

    # 5. merge
    staged.createOrReplaceTempView("staged")
    spark.sql("""
    MERGE INTO banking.gold.dim_customer AS t
    USING staged AS s
    ON t.customer_id = s.merge_key AND t.is_current = true
    WHEN MATCHED THEN UPDATE SET t.is_current = false, t.valid_to = s.updated_at
    WHEN NOT MATCHED THEN INSERT (
        customer_id, first_name, last_name, email, phone, city, province,
        is_id_conflict, valid_from, valid_to, is_current, customer_sk
    ) VALUES (
        s.customer_id, s.first_name, s.last_name, s.email, s.phone, s.city, s.province,
        false, s.updated_at, timestamp'9999-12-31 00:00:00', true,
        xxhash64(s.customer_id, s.updated_at)
    )
    """)

    print("dim rows now:", spark.table("banking.gold.dim_customer").count())

In [0]:
load_customer_batch("/Volumes/banking/raw/landing/batch_3/customers.csv")

+-----------+-----+
|change_type|count|
+-----------+-----+
|    CHANGED|  600|
|        NEW|  150|
+-----------+-----+

dim rows now: 6150


**Result:** 600 staged rows for 300 actual movers (two copies each) plus 150 new customers —
dimension grows from 5,700 to 6,150.

Customers who moved in *both* batch 2 and batch 3 now hold three versions. The
`is_current = true` predicate in the MERGE join ensures only the latest row is closed and
existing history is never rewritten.

In [0]:
from pyspark.sql import functions as F

dim = spark.table("banking.gold.dim_customer")

twice = (dim.groupBy("customer_id").count()
            .filter(F.col("count") == 3)
            .first()["customer_id"])

(dim.filter(F.col("customer_id") == twice)
    .select("customer_id", "city", "valid_from", "valid_to", "is_current")
    .orderBy("valid_from")
    .show(truncate=False))

+-----------+-------+-------------------+-------------------+----------+
|customer_id|city   |valid_from         |valid_to           |is_current|
+-----------+-------+-------------------+-------------------+----------+
|C001414    |CALGARY|2024-01-14 13:00:00|2024-02-15 00:00:00|false     |
|C001414    |OTTAWA |2024-02-15 00:00:00|2024-03-16 09:00:00|false     |
|C001414    |HALIFAX|2024-03-16 09:00:00|9999-12-31 00:00:00|true      |
+-----------+-------+-------------------+-------------------+----------+



## 7. Fact table — point-in-time correct join

Each transaction is joined to the customer version valid **at the moment the transaction
occurred**, not the current one.

Without the date-range predicate a customer with three versions matches all three,
triplicating every transaction. The dimension is broadcast (6,150 rows) to avoid shuffling
2M fact rows.

In [0]:
from pyspark.sql import functions as F

txn  = spark.table("banking.silver.transactions")
acct = spark.table("banking.silver.accounts").select("account_id", "customer_id", "branch_id")
dim  = (spark.table("banking.gold.dim_customer")
          .select(F.col("customer_id").alias("d_customer_id"),
                  "customer_sk", "valid_from", "valid_to"))

fact = (
    txn
    .join(acct, "account_id", "left")
    .join(F.broadcast(dim),
          (F.col("customer_id") == F.col("d_customer_id")) &
          (F.col("txn_timestamp") >= F.col("valid_from")) &
          (F.col("txn_timestamp") <  F.col("valid_to")),
          "left")
    .drop("d_customer_id")
)

print("fact rows:", fact.count())
print("no customer version found:",
      fact.filter(F.col("customer_sk").isNull() & ~F.col("is_orphan")).count())

fact rows: 2000000
no customer version found: 343936


**Problem:** 343,936 transactions (17%) matched no customer version. The root cause must be proven, not guessed.

In [0]:
from pyspark.sql import functions as F

first_version = (spark.table("banking.gold.dim_customer")
                   .groupBy("customer_id")
                   .agg(F.min("valid_from").alias("first_from")))

missed = (fact.filter(F.col("customer_sk").isNull() & ~F.col("is_orphan"))
              .join(first_version, "customer_id", "left"))

missed.select(
    F.count("*").alias("missed"),
    F.sum(F.when(F.col("txn_timestamp") < F.col("first_from"), 1).otherwise(0)).alias("before_first_version")
).show()

+------+--------------------+
|missed|before_first_version|
+------+--------------------+
|343936|              343936|
+------+--------------------+



**Proven:** all 343,936 misses occurred *before* the customer's earliest `valid_from`.

The initial load derived `valid_from` from the CRM's `updated_at` — when the record was last
*edited*, not when the customer began existing. Transactions predating that edit had no
covering version.

**Fix:** set each customer's earliest version to a sentinel past date, the standard SCD Type 2
convention for initial loads — the oldest known state is the best available information for
all prior time.

In [0]:
spark.sql("""
MERGE INTO banking.gold.dim_customer AS t
USING (
    SELECT customer_id, MIN(valid_from) AS first_from
    FROM banking.gold.dim_customer
    GROUP BY customer_id
) AS f
ON t.customer_id = f.customer_id AND t.valid_from = f.first_from
WHEN MATCHED THEN UPDATE SET t.valid_from = timestamp'1900-01-01 00:00:00'
""")

print("done")

done


In [0]:
from pyspark.sql import functions as F

txn  = spark.table("banking.silver.transactions")
acct = spark.table("banking.silver.accounts").select("account_id", "customer_id", "branch_id")
dim  = (spark.table("banking.gold.dim_customer")
          .select(F.col("customer_id").alias("d_customer_id"),
                  "customer_sk", "valid_from", "valid_to"))

fact = (
    txn
    .join(acct, "account_id", "left")
    .join(F.broadcast(dim),
          (F.col("customer_id") == F.col("d_customer_id")) &
          (F.col("txn_timestamp") >= F.col("valid_from")) &
          (F.col("txn_timestamp") <  F.col("valid_to")),
          "left")
    .drop("d_customer_id", "valid_from", "valid_to")
)

print("fact rows:", fact.count())
print("no customer version found:",
      fact.filter(F.col("customer_sk").isNull() & ~F.col("is_orphan")).count())

fact rows: 2000000
no customer version found: 0


**Result: 0 unmatched.**

Two numbers validate the join simultaneously: 0 misses proves there are **no gaps** in the
validity ranges, and an unchanged row count of exactly 2,000,000 proves there are **no
overlaps** — an overlap would have duplicated fact rows.

In [0]:
fact_out = fact.select(
    "txn_id", "txn_timestamp", "txn_date",
    "account_id", "customer_sk", "branch_id",
    "amount", "txn_type", "merchant", "currency",
    "is_orphan", "is_invalid_amount"
)

(fact_out.write.mode("overwrite")
    .partitionBy("txn_date")
    .saveAsTable("banking.gold.fact_transaction"))

print("fact_transaction:", spark.table("banking.gold.fact_transaction").count())

fact_transaction: 2000000


## 8. Data quality gate

Seven automated checks run after each load. All results are reported before the gate raises,
so every failure is visible in one run rather than discovered one at a time.

A custom `DataQualityError` distinguishes a data failure from a code defect.

In [0]:
class DataQualityError(Exception):
    pass

In [0]:
from pyspark.sql import functions as F

def run_quality_gate():
    s_cust = spark.table("banking.silver.customers")
    s_txn  = spark.table("banking.silver.transactions")
    dim    = spark.table("banking.gold.dim_customer")
    fact   = spark.table("banking.gold.fact_transaction")

    total_txn = fact.count()

    checks = []

    def check(name, actual, passed):
        checks.append((name, str(actual), "PASS" if passed else "FAIL"))

    dupes = s_cust.count() - s_cust.select("customer_id").distinct().count()
    check("silver customer_id unique", dupes, dupes == 0)

    multi_current = (dim.filter(F.col("is_current")).groupBy("customer_id").count()
                        .filter(F.col("count") != 1).count())
    check("one current row per customer", multi_current, multi_current == 0)

    sk_dupes = dim.count() - dim.select("customer_sk").distinct().count()
    check("customer_sk unique", sk_dupes, sk_dupes == 0)

    silver_txn = s_txn.count()
    check("fact rows = silver rows", f"{total_txn} vs {silver_txn}", total_txn == silver_txn)

    unlinked = fact.filter(F.col("customer_sk").isNull() & ~F.col("is_orphan")).count()
    check("valid txns linked to customer", unlinked, unlinked == 0)

    orphan_rate = fact.filter(F.col("is_orphan")).count() / total_txn
    check("orphan rate < 1%", f"{orphan_rate:.2%}", orphan_rate < 0.01)

    invalid_rate = fact.filter(F.col("is_invalid_amount")).count() / total_txn
    check("invalid amount rate < 1%", f"{invalid_rate:.2%}", invalid_rate < 0.01)

    spark.createDataFrame(checks, ["check", "actual", "result"]).show(truncate=False)

    failed = [c[0] for c in checks if c[2] == "FAIL"]
    if failed:
        raise DataQualityError(f"{len(failed)} check(s) failed: {failed}")

    print("all checks passed")

In [0]:
run_quality_gate()

+-----------------------------+------------------+------+
|check                        |actual            |result|
+-----------------------------+------------------+------+
|silver customer_id unique    |0                 |PASS  |
|one current row per customer |0                 |PASS  |
|customer_sk unique           |0                 |PASS  |
|fact rows = silver rows      |2000000 vs 2000000|PASS  |
|valid txns linked to customer|0                 |PASS  |
|orphan rate < 1%             |0.50%             |PASS  |
|invalid amount rate < 1%     |0.50%             |PASS  |
+-----------------------------+------------------+------+

all checks passed


The gate was verified by temporarily lowering the orphan threshold to 0.1%, confirming it
reports FAIL and halts the pipeline. A safety mechanism never observed to fire is unverified.

## 9. Delta time travel and recovery

Delta retains previous versions, so a bad write can be rolled back without reprocessing.
The limit is **file retention**, not the log.

In [0]:
spark.sql("DESCRIBE HISTORY banking.gold.dim_customer").select(
    "version", "timestamp", "operation"
).show(truncate=False)


+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|10     |2026-09-21 23:17:51|OPTIMIZE                         |
|9      |2026-09-21 23:17:49|MERGE                            |
|8      |2026-09-21 23:15:22|OPTIMIZE                         |
|7      |2026-09-21 23:15:21|MERGE                            |
|6      |2026-09-21 22:44:24|OPTIMIZE                         |
|5      |2026-09-21 22:44:22|MERGE                            |
|4      |2026-09-21 22:31:57|MERGE                            |
|3      |2026-09-21 22:31:33|CREATE OR REPLACE TABLE AS SELECT|
|2      |2026-09-21 22:25:16|MERGE                            |
|1      |2026-09-21 21:55:49|MERGE                            |
|0      |2026-09-21 20:28:01|CREATE OR REPLACE TABLE AS SELECT|
+-------+-------------------+---------------------------------+



### 9.1 Simulating a destructive write

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE banking.gold.dim_customer AS
SELECT * FROM banking.gold.dim_customer WHERE customer_id = 'NOBODY'
""")

print("rows now:", spark.table("banking.gold.dim_customer").count())

rows now: 0


In [0]:
spark.sql("SELECT COUNT(*) FROM banking.gold.dim_customer VERSION AS OF 10").show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4585376758368365>, line 1
----> 1 spark.sql("SELECT COUNT(*) FROM banking.gold.dim_customer VERSION AS OF 10").show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1556, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1554     req.user_context.user_id = self._user_id
  

**Time travel failed.** `DESCRIBE HISTORY` still lists version 10, but its data files were
removed after the default 7-day `deletedFileRetentionDuration` elapsed.

**The lesson:** the transaction log and the data files have separate lifetimes. A version
appearing in history does **not** mean it is readable. Longer recovery windows require raising
retention — and paying for the storage.

Recovery here came from the medallion architecture instead: Gold is derived data, so it was
rebuilt from Silver and the raw Volume. That is precisely why Bronze exists.

In [0]:
spark.sql("DESCRIBE HISTORY banking.gold.dim_customer").select(
    "version", "timestamp", "operation"
).show(5, truncate=False)

spark.sql("SHOW TBLPROPERTIES banking.gold.dim_customer").show(truncate=False)

+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|11     |2026-09-28 23:25:11|CREATE OR REPLACE TABLE AS SELECT|
|10     |2026-09-21 23:17:51|OPTIMIZE                         |
|9      |2026-09-21 23:17:49|MERGE                            |
|8      |2026-09-21 23:15:22|OPTIMIZE                         |
|7      |2026-09-21 23:15:21|MERGE                            |
+-------+-------------------+---------------------------------+
only showing top 5 rows
+-----------------------------------------+------------------------------------+
|key                                      |value                               |
+-----------------------------------------+------------------------------------+
|delta.enableDeletionVectors              |true                                |
|delta.feature.appendOnly                 |supported                        

### 9.2 Rollback on a fresh clone

`DEEP CLONE` copies data into new files, so the clone's versions are seconds old and well
inside the retention window.

In [0]:
spark.sql("CREATE OR REPLACE TABLE banking.gold.dim_test DEEP CLONE banking.gold.dim_customer")
print("rows:", spark.table("banking.gold.dim_test").count())

rows: 6150


In [0]:
spark.sql("DELETE FROM banking.gold.dim_test WHERE city = 'OTTAWA'")
print("rows after bad delete:", spark.table("banking.gold.dim_test").count())

rows after bad delete: 5155


In [0]:
spark.sql("DESCRIBE HISTORY banking.gold.dim_test").select("version", "operation").show()

spark.sql("SELECT COUNT(*) AS rows_at_v0 FROM banking.gold.dim_test VERSION AS OF 0").show()

+-------+---------+
|version|operation|
+-------+---------+
|      2| OPTIMIZE|
|      1|   DELETE|
|      0|    CLONE|
+-------+---------+

+----------+
|rows_at_v0|
+----------+
|      6150|
+----------+



The old version is confirmed readable **before** restoring — a rollback to an already-broken version helps nobody.

In [0]:
spark.sql("RESTORE TABLE banking.gold.dim_test TO VERSION AS OF 0")
print("rows after restore:", spark.table("banking.gold.dim_test").count())

spark.sql("DESCRIBE HISTORY banking.gold.dim_test").select("version", "operation").show()

rows after restore: 6150
+-------+---------+
|version|operation|
+-------+---------+
|      3|  RESTORE|
|      2| OPTIMIZE|
|      1|   DELETE|
|      0|    CLONE|
+-------+---------+



`RESTORE` appends a new version containing the old data rather than erasing history, so both
the mistake (version 1, DELETE) and the recovery (version 3, RESTORE) remain auditable.

## 10. Streaming ingestion with Auto Loader

Auto Loader tracks which files it has already processed in a **checkpoint**, making
reruns idempotent without any bookkeeping code.

The same Structured Streaming engine backs a Kafka source; switching source changes
configuration, not the processing model.

In [0]:
import csv, os, random
from datetime import datetime, timedelta

OUT = "/Volumes/banking/raw/landing/stream"
os.makedirs(OUT, exist_ok=True)
random.seed(99)

acct_ids = [r["account_id"] for r in
            spark.table("banking.silver.accounts").select("account_id").limit(500).collect()]

TXN_TYPES = ["PURCHASE","WITHDRAWAL","DEPOSIT","TRANSFER","PAYMENT"]
MERCHANTS = ["Amazon","Tim Hortons","Loblaws","Shell","Costco"]
BASE = datetime(2024, 4, 1)

def write_file(n, start_id, day):
    rows = []
    for i in range(n):
        ts = BASE + timedelta(days=day, hours=random.randint(0,23), minutes=random.randint(0,59))
        rows.append({
            "txn_id": f"S{start_id+i:09d}",
            "account_id": random.choice(acct_ids),
            "txn_timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
            "amount": round(random.lognormvariate(3.2, 1.1), 2),
            "txn_type": random.choice(TXN_TYPES),
            "merchant": random.choice(MERCHANTS),
            "currency": "CAD",
        })
    path = f"{OUT}/txn_day{day}.csv"
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    return path

for day in range(3):
    print(write_file(1000, day*1000 + 1, day))

/Volumes/banking/raw/landing/stream/txn_day0.csv
/Volumes/banking/raw/landing/stream/txn_day1.csv
/Volumes/banking/raw/landing/stream/txn_day2.csv


In [0]:
from pyspark.sql import functions as F

SRC        = "/Volumes/banking/raw/landing/stream"
CHECKPOINT = "/Volumes/banking/raw/landing/_chk/txn_stream"
SCHEMA_LOC = "/Volumes/banking/raw/landing/_schema/txn_stream"

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .option("header", "true")
    .load(SRC)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
)

query = (
    stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable("banking.bronze.transactions_stream")
)

query.awaitTermination()
print("rows:", spark.table("banking.bronze.transactions_stream").count())

rows: 4000


Rerunning the identical cell produces **no duplicates** — the checkpoint records all files as
processed. In batch, a repeated append would have doubled the table.

Adding a fourth file and rerunning picks up only the new file.

In [0]:
print(write_file(1000, 3001, 3))

/Volumes/banking/raw/landing/stream/txn_day3.csv


### 10.1 Deleting the checkpoint

The checkpoint is the stream's memory. Removing it makes Auto Loader treat every file as new.

In [0]:
dbutils.fs.rm("/Volumes/banking/raw/landing/_chk/txn_stream", recurse=True)
print("checkpoint deleted")

checkpoint deleted


In [0]:
from pyspark.sql import functions as F

t = spark.table("banking.bronze.transactions_stream")
print("total rows:  ", t.count())
print("distinct ids:", t.select("txn_id").distinct().count())

t.groupBy("txn_id").count().filter(F.col("count") > 1).show(3)


total rows:   8000
distinct ids: 4000
+----------+-----+
|    txn_id|count|
+----------+-----+
|S000001002|    2|
|S000001012|    2|
|S000001014|    2|
+----------+-----+
only showing top 3 rows


**Result: 8,000 rows, 4,000 distinct — every row duplicated exactly twice.**

The checkpoint is not an optimisation; it is the only thing preventing reprocessing. In
production it is critical state: one per stream, never shared between streams, never cleaned
up casually. Common causes of exactly this outage are a redeployment that changes the
checkpoint path, or dev and prod sharing one.

In [0]:
spark.sql("TRUNCATE TABLE banking.bronze.transactions_stream")
dbutils.fs.rm("/Volumes/banking/raw/landing/_chk/txn_stream", recurse=True)
print("reset")

reset


### 10.2 Watermarks and late-arriving events

Event time (when a transaction happened) and processing time (when the pipeline saw it)
diverge in streaming. Without a bound, windowed aggregations keep every window open forever
in case a late event arrives, and state grows until the job fails.

A **watermark** sets how late is too late. The file below contains 900 on-time events and 100
stamped three days earlier.

In [0]:
import csv, random
from datetime import datetime, timedelta

OUT = "/Volumes/banking/raw/landing/stream"
random.seed(123)

acct_ids = [r["account_id"] for r in
            spark.table("banking.silver.accounts").select("account_id").limit(500).collect()]

BASE = datetime(2024, 4, 4)
rows = []

# 900 on-time events on Apr 4
for i in range(900):
    ts = BASE + timedelta(hours=random.randint(0,23), minutes=random.randint(0,59))
    rows.append((f"L{i:09d}", ts))

# 100 LATE events — stamped 3 days earlier, Apr 1
for i in range(900, 1000):
    ts = BASE - timedelta(days=3, hours=random.randint(0,23))
    rows.append((f"L{i:09d}", ts))

out = []
for txn_id, ts in rows:
    out.append({
        "txn_id": txn_id,
        "account_id": random.choice(acct_ids),
        "txn_timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "amount": round(random.lognormvariate(3.2, 1.1), 2),
        "txn_type": random.choice(["PURCHASE","DEPOSIT","PAYMENT"]),
        "merchant": random.choice(["Amazon","Shell","Costco"]),
        "currency": "CAD",
    })

path = f"{OUT}/txn_late.csv"
with open(path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(out[0].keys()))
    w.writeheader()
    w.writerows(out)

print("wrote", path, len(out), "rows (100 are 3 days late)")

wrote /Volumes/banking/raw/landing/stream/txn_late.csv 1000 rows (100 are 3 days late)


In [0]:
from pyspark.sql import functions as F

SRC        = "/Volumes/banking/raw/landing/stream"
SCHEMA_LOC = "/Volumes/banking/raw/landing/_schema/txn_agg"
CHECKPOINT = "/Volumes/banking/raw/landing/_chk/txn_agg"

events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .option("header", "true")
    .load(SRC)
    .withColumn("event_time", F.to_timestamp("txn_timestamp", "yyyy-MM-dd HH:mm:ss"))
)

windowed = (
    events
    .withWatermark("event_time", "1 day")
    .groupBy(F.window("event_time", "1 day").alias("w"))
    .agg(F.count("*").alias("txn_count"))
    .select(F.col("w.start").alias("day"), "txn_count")
)

q = (windowed.writeStream
     .format("delta")
     .outputMode("append")
     .option("checkpointLocation", CHECKPOINT)
     .trigger(availableNow=True)
     .toTable("banking.gold.daily_txn_counts"))

q.awaitTermination()
spark.table("banking.gold.daily_txn_counts").orderBy("day").show()

+-------------------+---------+
|                day|txn_count|
+-------------------+---------+
|2024-03-31 00:00:00|       97|
|2024-04-01 00:00:00|     1003|
|2024-04-02 00:00:00|     1000|
+-------------------+---------+



Only three days appear. In `append` mode a window is written once the watermark passes its
end, so the most recent days are still open — held in memory pending late arrivals. That is
the completeness-versus-latency trade made visible.

Now events far beyond the watermark:

In [0]:
import csv, random
from datetime import datetime, timedelta

OUT = "/Volumes/banking/raw/landing/stream"
acct_ids = [r["account_id"] for r in
            spark.table("banking.silver.accounts").select("account_id").limit(100).collect()]

rows = []
for i in range(200):
    ts = datetime(2024, 3, 25) + timedelta(hours=random.randint(0, 23))
    rows.append({
        "txn_id": f"V{i:09d}",
        "account_id": random.choice(acct_ids),
        "txn_timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "amount": 50.0,
        "txn_type": "PURCHASE",
        "merchant": "Amazon",
        "currency": "CAD",
    })

path = f"{OUT}/txn_very_late.csv"
with open(path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print("wrote 200 rows stamped 2024-03-25 — 10 days behind the stream")

wrote 200 rows stamped 2024-03-25 — 10 days behind the stream


Rerunning the aggregation produces **no new window** — all 200 rows were dropped silently.
No error, no warning, and the daily counts still look complete.

A watermark is a deliberate decision to lose data, so it must be paired with monitoring. The
side-channel below captures stragglers to their own table rather than letting them vanish —
the same instinct as quarantining ID collisions and flagging orphans.

In [0]:
from pyspark.sql import functions as F

events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/banking/raw/landing/_schema/txn_late_side")
    .option("header", "true")
    .load("/Volumes/banking/raw/landing/stream")
    .withColumn("event_time", F.to_timestamp("txn_timestamp", "yyyy-MM-dd HH:mm:ss"))
)

late = events.filter(F.col("event_time") < F.lit("2024-03-30").cast("timestamp"))

q = (late.writeStream
     .format("delta")
     .option("checkpointLocation", "/Volumes/banking/raw/landing/_chk/txn_late_side")
     .trigger(availableNow=True)
     .toTable("banking.bronze.late_events"))

q.awaitTermination()
print("late events captured:", spark.table("banking.bronze.late_events").count())

late events captured: 200


## 11. Change Data Feed — incremental downstream processing

Rebuilding Gold reads every row even when a handful of customers changed. CDF records each
insert, update and delete as its own row with before and after images, so downstream jobs read
only what changed.

Unlike an `updated_at` column, CDF captures deletes and the previous value. This is Delta's
built-in CDC. It is **not retroactive** — only changes after enabling are recorded.

In [0]:
spark.sql("""
ALTER TABLE banking.gold.dim_customer
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql("DESCRIBE HISTORY banking.gold.dim_customer").select("version", "operation").show(3)

+-------+-----------------+
|version|        operation|
+-------+-----------------+
|     19|SET TBLPROPERTIES|
|     18|         OPTIMIZE|
|     17|            MERGE|
+-------+-----------------+
only showing top 3 rows


In [0]:
spark.sql("""
UPDATE banking.gold.dim_customer
SET city = 'WINNIPEG'
WHERE customer_id IN (
    SELECT customer_id FROM banking.gold.dim_customer
    WHERE is_current = true LIMIT 5
) AND is_current = true
""")

print("done")

done


In [0]:
from pyspark.sql import functions as F

changes = (spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 20)
    .table("banking.gold.dim_customer"))

(changes.select("customer_id", "city", "_change_type", "_commit_version")
        .orderBy("customer_id", "_change_type")
        .show(truncate=False))

+-----------+---------+----------------+---------------+
|customer_id|city     |_change_type    |_commit_version|
+-----------+---------+----------------+---------------+
|C001607    |WINNIPEG |update_postimage|20             |
|C001607    |HALIFAX  |update_preimage |20             |
|C001885    |WINNIPEG |update_postimage|20             |
|C001885    |MONTREAL |update_preimage |20             |
|C003739    |WINNIPEG |update_postimage|20             |
|C003739    |VANCOUVER|update_preimage |20             |
|C004232    |WINNIPEG |update_postimage|20             |
|C004232    |VANCOUVER|update_preimage |20             |
|C004936    |WINNIPEG |update_postimage|20             |
|C004936    |VANCOUVER|update_preimage |20             |
+-----------+---------+----------------+---------------+



`update_preimage` and `update_postimage` pair the before and after state of each change, so
you know not just *that* a row changed but *what it changed from*.

Using the feed to scope a downstream rebuild:

In [0]:
from pyspark.sql import functions as F

changed_customers = (spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 20)
    .table("banking.gold.dim_customer")
    .filter(F.col("_change_type").isin("update_postimage", "insert"))
    .select("customer_id").distinct())

print("customers to reprocess:", changed_customers.count())

affected = (spark.table("banking.gold.fact_transaction")
    .join(spark.table("banking.silver.accounts").select("account_id", "customer_id"), "account_id")
    .join(changed_customers, "customer_id"))

print("fact rows affected:", affected.count())
print("total fact rows:   ", spark.table("banking.gold.fact_transaction").count())

customers to reprocess: 5
fact rows affected: 3226
total fact rows:    2000000


**Result: 3,226 affected fact rows out of 2,000,000 — 0.16%.**

Reading the change feed to scope reprocessing reduced the footprint by three orders of
magnitude versus a full rebuild.

## 12. Governance — Unity Catalog

Unity Catalog provides one permission layer across workspaces with a three-level namespace
(`catalog.schema.table`), automatic column-level lineage, and attribute-based masking.

Permissions are **hierarchical**: `SELECT` on a table does nothing without `USE SCHEMA` and
`USE CATALOG` above it. A missing `USE SCHEMA` is the most common cause of "I granted access
but they still can't see it".

In [0]:
spark.sql("SHOW GRANTS ON TABLE banking.gold.dim_customer").show(truncate=False)
spark.sql("SHOW GRANTS ON SCHEMA banking.gold").show(truncate=False)

+---------+----------+----------+---------+
|Principal|ActionType|ObjectType|ObjectKey|
+---------+----------+----------+---------+
+---------+----------+----------+---------+

+---------+----------+----------+---------+
|Principal|ActionType|ObjectType|ObjectKey|
+---------+----------+----------+---------+
+---------+----------+----------+---------+



In [0]:
me = spark.sql("SELECT current_user()").first()[0]
print("user:", me)

spark.sql(f"GRANT USE CATALOG ON CATALOG banking TO `{me}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA banking.gold TO `{me}`")
spark.sql(f"GRANT SELECT ON TABLE banking.gold.dim_customer TO `{me}`")

spark.sql("SHOW GRANTS ON TABLE banking.gold.dim_customer").show(truncate=False)

user: hinalvasava2002@gmail.com
+-------------------------+----------+----------+-------------------------+
|Principal                |ActionType|ObjectType|ObjectKey                |
+-------------------------+----------+----------+-------------------------+
|hinalvasava2002@gmail.com|SELECT    |TABLE     |banking.gold.dim_customer|
+-------------------------+----------+----------+-------------------------+



Grants are normally issued to **groups**, not individuals, so adding a person is one
membership change rather than many grants. Group creation is an account-level operation
(console or API), not SQL — identity management sits above any single workspace.

Lineage is derived automatically from query history, so it cannot drift out of date the way a
hand-drawn diagram does. For `dim_customer` it shows `silver.customers` and the raw Volume
upstream, with `fact_transaction` downstream — the basis for impact analysis before changing
an upstream table.

## 13. Result — quantifying the value of SCD Type 2

Regional spend computed two ways: joined to the customer's city **at transaction time**
(correct) versus their **current** city (what overwriting would produce).

| City | Point-in-time | If overwritten | Difference |
|---|---:|---:|---:|
| Calgary | 15,328,112 | 15,420,520 | +92,408 |
| Halifax | 15,071,866 | 14,919,803 | −152,063 |
| **Montreal** | **14,700,750** | **15,348,946** | **+648,196** |
| Ottawa | 14,624,658 | 14,733,581 | +108,923 |
| **Toronto** | **14,799,011** | **14,437,816** | **−361,195** |
| Vancouver | 14,396,806 | 14,060,537 | −336,269 |

**The differences sum to zero.** Overwriting does not lose revenue — it misattributes it
between regions while the grand total still reconciles, making the error invisible to
total-based validation.

Montreal would be overstated by **$648k (4.4%)** and Toronto understated by **$361k**.
Regional budgets and performance reviews would be set on fictional numbers.

This is the concrete business case for SCD Type 2.

In [0]:
from pyspark.sql import functions as F

fact = spark.table("banking.gold.fact_transaction").filter(~F.col("is_orphan") & ~F.col("is_invalid_amount"))
dim  = spark.table("banking.gold.dim_customer")

# right: city at the time of the transaction
point_in_time = (fact.join(dim.select("customer_sk", "customer_id", "city"), "customer_sk")
                     .groupBy("city").agg(F.round(F.sum("amount"), 0).alias("spend_correct")))

# wrong: everyone's current city
current = dim.filter(F.col("is_current")).select("customer_id", F.col("city").alias("current_city"))
overwritten = (fact.join(dim.select("customer_sk", "customer_id"), "customer_sk")
                   .join(current, "customer_id")
                   .groupBy(F.col("current_city").alias("city"))
                   .agg(F.round(F.sum("amount"), 0).alias("spend_if_overwritten")))

(point_in_time.join(overwritten, "city")
    .withColumn("difference", F.col("spend_if_overwritten") - F.col("spend_correct"))
    .orderBy("city")
    .show())

+---------+-------------+--------------------+----------+
|     city|spend_correct|spend_if_overwritten|difference|
+---------+-------------+--------------------+----------+
|  CALGARY|  1.5328112E7|          1.542052E7|   92408.0|
|  HALIFAX|  1.5071866E7|         1.4919803E7| -152063.0|
| MONTREAL|   1.470075E7|         1.5348946E7|  648196.0|
|   OTTAWA|  1.4624658E7|         1.4733581E7|  108923.0|
|  TORONTO|  1.4799011E7|         1.4437816E7| -361195.0|
|VANCOUVER|  1.4396806E7|         1.4060537E7| -336269.0|
+---------+-------------+--------------------+----------+



## Summary

| Layer | Tables | Purpose |
|---|---|---|
| **Bronze** | customers, accounts, transactions, branches, transactions_stream | Raw as-landed + audit columns; enables replay |
| **Silver** | customers, customer_conflicts, accounts, transactions, branches | One row per entity, typed, standardised, flagged |
| **Gold** | dim_customer, fact_transaction, daily_txn_counts | SCD Type 2 history, point-in-time correct star schema |

### Data quality findings

| Finding | Volume | Resolution |
|---|---:|---|
| Customer ID collisions | 250 (5%) | Latest kept and flagged; loser quarantined |
| Orphan transactions | 10,035 (0.5%) | Flagged, retained — likely late-arriving dimension |
| Invalid amounts | 9,969 (0.5%) | Flagged for consumer filtering |
| Missing emails | 667 (13.3%) | Reported as a metric; empty strings → NULL |
| Mixed date formats | 1,078 (15%) | Multi-format parse, coverage proven at 0 failures |
| Transactions predating first version | 343,936 (17%) | Sentinel `valid_from` on initial load |
| Duplication after checkpoint loss | 4,000 (2×) | Demonstrated, then reset — checkpoint is critical state |
| Events beyond watermark | 200 | Routed to a late-events side table |

### Engineering decisions

- **Flag, don't drop.** Deleting orphans permanently loses transactions merely waiting on a
  late-arriving account.
- **Never invent values.** `'unknown'` looks like data and defeats null checks; NULL is honest.
- **Survivorship requires confirmed identity.** Borrowing fields across an ID collision
  fabricates a person who does not exist.
- **Prove, don't assume.** Format coverage, uniqueness, and join completeness were each
  verified with an explicit before/after count.
- **Fail loudly on systemic problems, skip quietly on individual bad rows.** A rate threshold
  is the boundary between the two.
- **Time travel is not a backup.** Retention bounds it; the medallion architecture is what
  actually enabled recovery.